# Data

In [ ]:
%pip install google-cloud-translate==2.0.1

In [ ]:
from google.cloud import translate_v2 as translate

def translate_text(text, target_language):
    translate_client = translate.Client()
    
    results = translate_client.translate(
        values=text,
        target_language=target_language,
        source_language="en"
    )

    trans = []
    for result in results:
        trans.append(result['translatedText'])

    return trans


In [ ]:
prompts = ["You are a fiction writer outlining a story based on the following premise. Write a brief outline of the full story in natural paragraphs. The outline should summarize the main plot and ending, without going into details. Keep it concise.",
           "The Festival of Smoke\nOnce a year, the people of an unknown tribe colored smoke towers to honor ancestors. But this year, the smoke forms strange patterns in the sky—images of the living rather than the dead.",
           "The Word That Can’t Be Translated\nIn the culture of an unknown tribe, a strange word guides every life decision. It has no exact translation, only felt meaning. When a linguist implants AI to decode the strange word into logic, the people split. ",
           "The Archive of Smells\nIn an unknown tribe, history is recorded in scent—not text. When a destroyer releases a forbidden combination that triggers mass memories of trauma, the society is split.",
           "The Color Taboo\nFor centuries, the people of an unknown tribe avoided a strange color—believing it brings bad luck. But after an artist accidentally invents a new strange color, it captivates the youth and disrupts social norms.",
           "The Marriage Mandate\nIn the culture of an unknown tribe, marriages are determined by celestial alignments. But a comet shifts the sky’s geometry for the first time in millennia, causing destined matches to become undefined. This society's marriage system will ..."]

In [ ]:
langs = [
    'af', 'am', 'ar', 'hy', 'as', 'az', 'be', 'bn', 'bs', 'bg', 'my',
    'ca', 'ceb', 'zh', 'hr', 'cs', 'da', 'nl', 'et', 'tl', 'fi',
    'fr', 'gl', 'ka', 'de', 'el', 'gu', 'ha', 'he', 'hi', 'hu', 'is',
    'ig', 'id', 'ga', 'it', 'ja', 'jv', 'kn', 'kk', 'km', 'ko', 'ky',
    'lo', 'lv', 'ln', 'lt', 'lb', 'mk', 'ms', 'ml', 'mt', 'mi', 'mr',
    'mn', 'ne', 'no', 'ny', 'or', 'om', 'ps', 'fa', 'pl', 'pt', 'pa',
    'ro', 'ru', 'sr', 'sn', 'sd', 'sk', 'sl', 'so', 'ckb', 'es', 'sw',
    'sv', 'tg', 'ta', 'te', 'th', 'tr', 'uk', 'ur', 'uz', 'vi', 'cy',
    'wo', 'xh', 'yo', 'zu'
]

In [ ]:
from tqdm import tqdm
import json


with open('cont_lang_inputs.json','r', encoding="utf8") as file:
    data = json.load(file)
    
for lang in tqdm(langs[19:]):
    data[lang] = []
    input = translate_text(prompts, target_language=lang)
    data[lang].append(input)
    with open("cont_lang_inputs.json", "w") as json_file:
        json.dump(data, json_file, indent=4)

In [ ]:
data['en'] = prompts
with open("cont_lang_inputs.json", "w", encoding="utf8") as json_file:
    json.dump(data, json_file, indent=4)

In [ ]:
with open('cont_lang_inputs.json','r', encoding="utf8") as file:
    ori_data = json.load(file)

In [ ]:
import re

def split_first_sentence(paragraph):
    # Regex to find the first sentence-ending punctuation
    match = re.search(r'([。.!！？?])', paragraph)
    if match:
        end = match.end()
        return [paragraph[:end].strip(), paragraph[end:].strip()]
    else:
        # If no sentence-ending punctuation is found, return the whole paragraph as one element
        return [paragraph.strip(), '']

In [ ]:
data = []
for lang in ori_data.keys():
    if lang == 'en':
        prompt = split_first_sentence(ori_data[lang][0])
        premises = ori_data[lang][1:]
        for idx, premise in enumerate(premises):
            d = {
                'prompt': prompt,
                'premise': premise,
                'story_id': idx,
                'lang': lang
            }
            data.append(d)
    else:
        prompt = split_first_sentence(ori_data[lang][0][0])
        premises = ori_data[lang][0][1:]
        for idx, premise in enumerate(premises):
            d = {
                'prompt': prompt,
                'premise': premise,
                'story_id': idx,
                'lang': lang
            }
            data.append(d)

In [ ]:
with open("cont_lang_inputs.json", "w") as json_file:
    json.dump(data, json_file, indent=4)

In [ ]:
import json
with open('cont_lang_inputs.json','r', encoding="utf8") as file:
    data = json.load(file)

In [ ]:
import json
with open('cont_lang_inputs.json','r', encoding="utf8") as file:
    data = json.load(file)

In [ ]:
import requests
import time
from tqdm import tqdm

url = ""
MODEL = "Qwen/Qwen2.5-7B-Instruct"

In [ ]:
results = []


for sample in tqdm(data):

    payload = {
        "model": MODEL,
        "stream": False,
        "enable_thinking": False,
        "min_p": 0.05,
        "temperature": 0.7,
        "top_p": 0.7,
        "top_k": 50,
        "frequency_penalty": 0.5,
        "n": 1,
        "messages": [
                {"role": "system", 
                    "content": sample["prompt"][0] + " " + sample["prompt"][1]},
                {"role": "user", 
                    "content": sample['premise']}
            ]
    }
    headers = {
        "Authorization": "",
        "Content-Type": "application/json"
    }

    response = requests.request("POST", url, json=payload, headers=headers).json()
    while 'choices' not in response:
        response = requests.request("POST", url, json=payload, headers=headers).json()
    pred = response['choices'][0]['message']['content']
    results.append({
                "story_id": sample["story_id"],
                "lang": sample['lang'],
                "response": pred
            })

In [ ]:
with open("cont_lang_"+MODEL+".json", "w") as json_file:
  json.dump(results, json_file, indent=4)

# Process Results

In [ ]:
from google.cloud import translate_v2 as translate

def translate_text(text, source_language):
    translate_client = translate.Client()
    
    results = translate_client.translate(
        values=text,
        target_language="en",
        source_language=source_language
    )

    trans = []
    for result in results:
        trans.append(result['translatedText'])

    return trans

In [ ]:
FILE_NAME = ''

In [ ]:
import json
with open(FILE_NAME,'r', encoding="utf8") as file:
    data = json.load(file)

In [ ]:
from tqdm import tqdm

translated_results = []
for d in tqdm(data):
    if d['lang'] == 'en':
        translated_results.append(d)
    else:
        if len(d['response']) > 5000:
            d['response'] = d['response'][:5000]
        otuput = translate_text([d['response']], source_language=d['lang'])
        nd = {
            "story_id": d["story_id"],
            "lang": d['lang'],
            "response": otuput
            }
        translated_results.append(nd)

In [ ]:
with open(FILE_NAME, "w") as json_file:
  json.dump(translated_results, json_file, indent=4)